# Parameter Sensitivity Analysis

This notebook reproduces Table S3 for one-at-a-time sensitivity analyses under representative extreme heat and cold scenarios. MRT is calculated with `Utils.climateProcess.calc_MRT`, and core-temperature changes use the same energy conversion as the panel preprocessing scripts.

In [88]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Utils":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import Utils.climateProcess as cp
import Utils.rider_character as rider_character

OUTPUT_DIR = PROJECT_ROOT / "Utils"
OUTPUT_CSV = OUTPUT_DIR / "TableS3_Parameter_sensitivity_analyses.csv"
OUTPUT_XLSX = OUTPUT_DIR / "TableS3_Parameter_sensitivity_analyses.xlsx"

SPECIFIC_HEAT_BODY = 2980.0
ACTIVE_HOURS = 0.8
REST_HOURS = 0.2

## Scenario Settings

In [89]:
HEAT_SCENARIO = {
    "Ta_C": 38,
    "RH": 40.0,
    "SWR": 800.0,
    "cloud_fraction": 0.0,
    "bowen_ratio": 1.0,
    "svf": 0.6,
    "cycling_speed_ms": 7.0,
    "summer_clo": 0.36,
    "w_max": 0.85,
}

COLD_SCENARIO = {
    "Ta_C": -30.0,
    "RH": 70.0,
    "SWR": 0.0,
    "cloud_fraction": 1.0,
    "bowen_ratio": 3.0,
    "svf": 0.6,
    "cycling_speed_ms": 7.0,
    "winter_clo": 2.4,
}

BASELINE_PERSON = {
    "height": 1.70,
    "mass": 70.0,
}

## Helper Functions

In [90]:
class SensitivityRider(rider_character.rider):
    def __init__(self, *args, icl_override=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.icl_override = icl_override

    def cal_Icl(self, Ta_C):
        if self.icl_override is not None:
            return self.icl_override
        return super().cal_Icl(Ta_C)


def format_deviation_range(values):
    if values is None:
        return "-"
    ordered_values = sorted(values)
    return f"[{ordered_values[0]:.2f}, {ordered_values[1]:.2f}]"


def make_worker(location, mass=70.0, height=1.70, w_max=0.85, icl_override=None):
    worker = SensitivityRider(location, "YNG_Morris_2021", icl_override=icl_override)
    worker.Mass = mass
    worker.Height = height
    worker.AD = worker.AD_from_mass_height()
    worker.w_max = w_max
    return worker


def calc_scenario_mrt(scenario):
    return float(
        cp.calc_MRT(
            Ta_C=scenario["Ta_C"],
            RH=scenario["RH"],
            N=scenario["cloud_fraction"],
            WS=0.0,
            SWR=scenario["SWR"],
            Bowen_ratio=scenario["bowen_ratio"],
            SVF=scenario["svf"],
        )
    )

In [91]:
def heat_core_temp_change(
    mass=BASELINE_PERSON["mass"],
    height=BASELINE_PERSON["height"],
    w_max=HEAT_SCENARIO["w_max"],
    summer_clo=HEAT_SCENARIO["summer_clo"],
    cycling_speed_ms=HEAT_SCENARIO["cycling_speed_ms"],
    svf=HEAT_SCENARIO["svf"],
):
    scenario = {**HEAT_SCENARIO, "svf": svf}
    worker = make_worker(
        location="Shanghai",
        mass=mass,
        height=height,
        w_max=w_max,
        icl_override=summer_clo,
    )
    mrt_c = calc_scenario_mrt(scenario)
    active_storage = worker.heat_storage_in_the_period(
        RH=scenario["RH"],
        Ta_C=scenario["Ta_C"],
        Av_ms=cycling_speed_ms,
        MRT_C=mrt_c,
        TimeExposure_hr=ACTIVE_HOURS,
    )

    worker.METS = 1.5
    resting_storage = worker.heat_storage_in_the_period(
        RH=50.0,
        Ta_C=25.0,
        Av_ms=0.5,
        MRT_C=25.0,
        TimeExposure_hr=REST_HOURS,
    )
    return (active_storage + resting_storage) / (mass * SPECIFIC_HEAT_BODY)


def cold_core_temp_change(
    mass=BASELINE_PERSON["mass"],
    height=BASELINE_PERSON["height"],
    winter_clo=COLD_SCENARIO["winter_clo"],
    cycling_speed_ms=COLD_SCENARIO["cycling_speed_ms"],
):
    worker = make_worker(
        location="Harbin",
        mass=mass,
        height=height,
        icl_override=winter_clo,
    )
    mrt_c = calc_scenario_mrt(COLD_SCENARIO)
    return worker.heat_dissipation_in_the_period(
        RH=COLD_SCENARIO["RH"],
        Ta_C=COLD_SCENARIO["Ta_C"],
        Av_ms=cycling_speed_ms,
        MRT_C=mrt_c,
        TimeExposure_hr=ACTIVE_HOURS,
    ) / (mass * SPECIFIC_HEAT_BODY)

## Run One-at-a-Time Sensitivity Analysis

In [92]:
baseline_heat = heat_core_temp_change()
baseline_cold = cold_core_temp_change()

sensitivity_rows = [
    {
        "Category": "Human factor",
        "Parameter": "Height",
        "Baseline": "1.70 m",
        "Range": "[1.65,1.75] m",
        "heat_values": [heat_core_temp_change(height=1.65) - baseline_heat, heat_core_temp_change(height=1.75) - baseline_heat],
        "cold_values": [cold_core_temp_change(height=1.65) - baseline_cold, cold_core_temp_change(height=1.75) - baseline_cold],
    },
    {
        "Category": "Human factor",
        "Parameter": "Body mass",
        "Baseline": "70 kg",
        "Range": "[65,75] kg",
        "heat_values": [heat_core_temp_change(mass=65.0) - baseline_heat, heat_core_temp_change(mass=75.0) - baseline_heat],
        "cold_values": [cold_core_temp_change(mass=65.0) - baseline_cold, cold_core_temp_change(mass=75.0) - baseline_cold],
    },
    {
        "Category": "Human factor",
        "Parameter": "Heat acclimation",
        "Baseline": "0.85",
        "Range": "[0.8,0.9]",
        "heat_values": [heat_core_temp_change(w_max=0.8) - baseline_heat, heat_core_temp_change(w_max=0.9) - baseline_heat],
        "cold_values": None,
    },
    {
        "Category": "Behavior",
        "Parameter": "Winter clothing",
        "Baseline": "2.4 clo",
        "Range": "[2.35,2.45] clo",
        "heat_values": None,
        "cold_values": [cold_core_temp_change(winter_clo=2.35) - baseline_cold, cold_core_temp_change(winter_clo=2.45) - baseline_cold],
    },
    {
        "Category": "Behavior",
        "Parameter": "Summer clothing",
        "Baseline": "0.36 clo",
        "Range": "[0.32,0.4] clo",
        "heat_values": [heat_core_temp_change(summer_clo=0.32) - baseline_heat, heat_core_temp_change(summer_clo=0.4) - baseline_heat],
        "cold_values": None,
    },
    {
        "Category": "Behavior",
        "Parameter": "Cycling speed",
        "Baseline": "7 m/s",
        "Range": "[6,8] m/s",
        "heat_values": [heat_core_temp_change(cycling_speed_ms=6.0) - baseline_heat, heat_core_temp_change(cycling_speed_ms=8.0) - baseline_heat],
        "cold_values": [cold_core_temp_change(cycling_speed_ms=6.0) - baseline_cold, cold_core_temp_change(cycling_speed_ms=8.0) - baseline_cold],
    },
    {
        "Category": "Urban",
        "Parameter": "Sky view factor",
        "Baseline": "0.6",
        "Range": "[0.5,0.7]",
        "heat_values": [heat_core_temp_change(svf=0.5) - baseline_heat, heat_core_temp_change(svf=0.7) - baseline_heat],
        "cold_values": None,
    },
]

table_s3 = pd.DataFrame(
    [
        {
            "Category": row["Category"],
            "Parameter": row["Parameter"],
            "Baseline": row["Baseline"],
            "Range": row["Range"],
            "Predicted deviation heat (deg C)": format_deviation_range(row["heat_values"]),
            "Predicted deviation cold (deg C)": format_deviation_range(row["cold_values"]),
            "Baseline heat Delta Tcore (deg C)": round(baseline_heat, 3),
            "Baseline cold Delta Tcore (deg C)": round(baseline_cold, 3),
        }
        for row in sensitivity_rows
    ]
)

table_s3

,Category,Parameter,Baseline,Range,Predicted deviation heat (deg C),Predicted deviation cold (deg C),Baseline heat Delta Tcore (deg C),Baseline cold Delta Tcore (deg C)
0,Human factor,Height,1.70 m,"[1.65,1.75] m","[-0.03, 0.03]","[-0.02, 0.02]",1.955,1.014
1,Human factor,Body mass,70 kg,"[65,75] kg","[-0.14, 0.12]","[-0.04, 0.04]",1.955,1.014
2,Human factor,Heat acclimation,0.85,"[0.8,0.9]","[-0.05, 0.05]",-,1.955,1.014
3,Behavior,Winter clothing,2.4 clo,"[2.35,2.45] clo",-,"[-0.12, 0.13]",1.955,1.014
4,Behavior,Summer clothing,0.36 clo,"[0.32,0.4] clo","[-0.14, 0.09]",-,1.955,1.014
5,Behavior,Cycling speed,7 m/s,"[6,8] m/s","[-0.08, 0.09]","[-0.33, 0.31]",1.955,1.014
6,Urban,Sky view factor,0.6,"[0.5,0.7]","[-0.27, 0.27]",-,1.955,1.014


## Save Outputs

In [93]:
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# table_s3.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
# table_s3.to_excel(OUTPUT_XLSX, index=False)

# print(f"Saved CSV: {OUTPUT_CSV}")
# print(f"Saved Excel: {OUTPUT_XLSX}")